# 9. MCP (Model Context Protocol) Client, Inside a Graph

The LangGraph-flavored counterpart to the `langchain` folder's MCP notebook: a
hand-built reason/act loop — an explicit two-node `StateGraph` (`call_model` <->
`call_tools`) — using tools fetched from this project's own MCP server, instead
of `langgraph.prebuilt.create_react_agent`. Building the loop by hand also lets
us add something a prebuilt can't easily offer: `call_tools` checks the
conversation so far for a tool already called with identical arguments and
reuses that result instead of re-invoking the tool — a real fix found while
building this project (a weaker model was observed to re-ask for the same tool
call several times even with a system-prompt instruction telling it not to).

**Prerequisites:** `scripts/start_mcp_server.sh` running (port `18383`), Ollama
running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

## Connect to the MCP server

In [ ]:
import os

from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_SERVER_URL = f"http://localhost:{os.getenv('MCP_SERVER_PORT', '18383')}/mcp"
mcp_client = MultiServerMCPClient({"ai_tutorial": {"transport": "streamable_http", "url": MCP_SERVER_URL}})

mcp_tools = await mcp_client.get_tools(server_name="ai_tutorial")
print([tool.name for tool in mcp_tools])

## Build the loop as an explicit `StateGraph`

In [ ]:
from langchain_core.messages import SystemMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

from models.chat_models.ollama_models import SupportedModel, get_chat_model

llm = get_chat_model(SupportedModel.llama3_2)

AGENT_SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. Once a tool call has "
    "returned a result that answers the user's question, respond directly "
    "with the final answer in plain text. Never call the same tool with the "
    "same arguments more than once."
)


def find_prior_tool_result(messages: list, name: str, args: dict) -> str | None:
    """Look for a tool call already made earlier with the same name/args,
    and return its already-computed result if found — a hard, deterministic
    backstop for a model that keeps re-asking for the same tool call."""
    for i, message in enumerate(messages):
        if getattr(message, "type", None) == "ai" and getattr(message, "tool_calls", None):
            for tool_call in message.tool_calls:
                if tool_call["name"] == name and tool_call["args"] == args:
                    for later in messages[i + 1 :]:
                        if getattr(later, "type", None) == "tool" and later.tool_call_id == tool_call["id"]:
                            return later.content
    return None


async def call_model(state: MessagesState) -> dict:
    response = llm.bind_tools(mcp_tools).invoke([SystemMessage(content=AGENT_SYSTEM_PROMPT), *state["messages"]])
    return {"messages": [response]}


async def call_tools(state: MessagesState) -> dict:
    tools_by_name = {tool.name: tool for tool in mcp_tools}
    last_message = state["messages"][-1]
    tool_messages = []
    for tool_call in last_message.tool_calls:
        cached_result = find_prior_tool_result(state["messages"][:-1], tool_call["name"], tool_call["args"])
        if cached_result is not None:
            result = cached_result
        else:
            tool = tools_by_name[tool_call["name"]]
            try:
                result = await tool.ainvoke(tool_call["args"])
            except Exception as exc:
                result = f"Error calling tool '{tool_call['name']}': {exc}"
        tool_messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))
    return {"messages": tool_messages}


def route_after_model(state: MessagesState):
    return "call_tools" if state["messages"][-1].tool_calls else END


graph = StateGraph(MessagesState)
graph.add_node("call_model", call_model)
graph.add_node("call_tools", call_tools)
graph.add_edge(START, "call_model")
graph.add_conditional_edges("call_model", route_after_model)
graph.add_edge("call_tools", "call_model")
compiled = graph.compile(checkpointer=InMemorySaver())

## Run it, multi-turn

In [ ]:
from langchain_core.messages import HumanMessage

thread = {"configurable": {"thread_id": "notebook-demo"}, "recursion_limit": 20}

result_1 = await compiled.ainvoke({"messages": [HumanMessage(content="what is 3+4?")]}, config=thread)
print(result_1["messages"][-1].content)

In [ ]:
result_2 = await compiled.ainvoke(
    {"messages": [HumanMessage(content="what happens when I add 5 to it?")]}, config=thread
)
print(result_2["messages"][-1].content)

In [ ]:
# Full step trace for the whole thread:
for message in result_2["messages"]:
    message_type = getattr(message, "type", None)
    if message_type == "ai" and getattr(message, "tool_calls", None):
        for tool_call in message.tool_calls:
            print(f"tool_call: {tool_call['name']}({tool_call['args']})")
    elif message_type == "tool":
        print(f"tool_result: {message.content}")
    elif message_type == "ai" and message.content:
        print(f"ai_message: {message.content}")

## 🧪 Playground

**1. Remove the `find_prior_tool_result` cache** (make `call_tools` always re-invoke) and try a harder multi-part question on a fresh thread — do you see repeated identical tool calls?

In [ ]:
# TODO: build a second graph without the cache check and compare


**2. Visualize the graph** — `print(compiled.get_graph().draw_mermaid())`.

In [ ]:
# TODO: draw_mermaid()


**3. A fresh thread, single-turn, multi-part question** — try `"what is 12 times 7, then add 5 to that?"` and inspect how many tool calls it takes.

In [ ]:
# TODO: new thread, multi-part question in one turn
